# AI & Margin Analysis — Full Pipeline
**ML Project SEA · A.Y. 2025/2026 · Alkemy**

| | |
|---|---|
| **Research question** | Beyond which AI usage threshold does rework destroy profit margin? |
| **Dataset** | `data/ai_productivity_dataset_final.csv` — 3,248 tasks, 34 variables |
| **Pipeline** | Phases 1–10: Cleaning → EDA → Threshold → Mediation → OLS → RF → Decision |

This notebook follows the project pipeline defined in `data/project_pipeline.md`. Every code cell is preceded by a markdown cell explaining the **what** and **why**.

---
## 0 — Imports & Setup

We load all libraries upfront and define the visual theme. A single colour palette is used throughout so all charts are immediately recognisable as belonging to the same project.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import ruptures as rpt
import pingouin as pg
import shap

warnings.filterwarnings('ignore')

# ── Dark-mode visual theme ─────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#3a3d4a',
    'axes.labelcolor': '#e0e0e0',
    'axes.titlecolor': '#ffffff',
    'xtick.color': '#a0a0a0',
    'ytick.color': '#a0a0a0',
    'text.color': '#e0e0e0',
    'grid.color': '#2a2d3a',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'legend.facecolor': '#1a1d27',
    'legend.edgecolor': '#3a3d4a',
})

SEA_BLUE   = '#7c6ae6'
SEA_RED    = '#e06c75'
SEA_GREEN  = '#98c379'
SEA_YELLOW = '#e5c07b'
SEA_CYAN   = '#56b6c2'
PALETTE    = [SEA_BLUE, SEA_RED, SEA_GREEN, SEA_YELLOW, SEA_CYAN]

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = os.path.abspath('')
PATH = os.path.join(BASE, 'data', 'ai_productivity_dataset_final.csv')
IMG  = os.path.join(BASE, 'images')
os.makedirs(IMG, exist_ok=True)

print(f'Data:   {PATH}')
print(f'Images: {IMG}')
print('Setup complete ✓')

---
## Phase 1 — Data Loading & Cleaning

The project brief warns the dataset is *"multi-table, incomplete, and imperfect."* We fix every known issue before any analysis: duplicates, team typos, string dates, boolean encoding, missing values, and extreme outliers. Any analysis built on dirty data will produce unreliable conclusions.

### 1.1 Load & Inspect

Print shape, dtypes, and head to understand the data before touching it.

In [ ]:
df_raw = pd.read_csv(PATH)
print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'\nColumn names:')
print(list(df_raw.columns))
print(f'\nData types:')
print(df_raw.dtypes.to_string())
df_raw.head(3)

In [ ]:
print('── Numeric Summary ──')
df_raw.describe().round(2)

### 1.2 Deduplication

Duplicate rows inflate AI usage counts and bias every downstream metric. We deduplicate on `task_id` — the natural primary key.

In [ ]:
df = df_raw.copy()
before = len(df)
df = df.drop_duplicates(subset='task_id')
after  = len(df)
print(f'Rows before: {before:,}')
print(f'Rows after:  {after:,}  ({before - after} duplicates removed)')

### 1.3 Team Normalisation

The `team` column has typos (`Contennt`, `Desgn`) and inconsistent casing. Without fixing this, team-level segmentation creates phantom groups that split the data incorrectly.

In [ ]:
print('Teams BEFORE:', sorted(df['team'].unique()))

team_map = {
    'seo': 'SEO', 'SEO ': 'SEO',
    'media': 'Media', 'MEDIA': 'Media', 'Paid Media': 'Media',
    'content': 'Content', 'CONTENT': 'Content', 'Contennt': 'Content',
    'design': 'Design', 'DESIGN': 'Design', 'Desgn': 'Design',
}
df['team'] = df['team'].replace(team_map)
print('Teams AFTER: ', sorted(df['team'].unique()))
print(f'\nDistribution:\n{df["team"].value_counts().to_string()}')

### 1.4 Date Parsing

Dates arrive as strings. We parse them to `datetime` so we can later compute `duration_days = delivered_at − created_at`. Invalid strings are coerced to `NaT`.

In [ ]:
for col in ['created_at', 'delivered_at', 'updated_at']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f'  {col}: parsed → {df[col].isna().sum()} invalid dates became NaT')

### 1.5 Legacy AI Flag

`legacy_ai_flag` contains `'true'`, `'false'`, and `'unknown'`. We map to proper booleans; `'unknown'` becomes `NaN` — treating it as True or False would be an arbitrary guess that could bias binary tests.

In [ ]:
print('Before:', df['legacy_ai_flag'].value_counts(dropna=False).to_string())
df['legacy_ai_flag'] = df['legacy_ai_flag'].map({'true': True, 'false': False})
print('After: ', df['legacy_ai_flag'].value_counts(dropna=False).to_string())

### 1.6–1.7 Missing Value Imputation

For numeric columns with < 5% missing we impute with the **median** — robust to outliers and right-skewed distributions unlike the mean.

For `billable_hours` we use `hours_spent × 0.85`. The **0.85 factor** reflects the industry-standard assumption that ~85% of worked hours are billable in professional services (the remaining 15% covers admin, internal reviews, and context-switching). This is a documented benchmark widely used in agency financial planning.

In [ ]:
num_impute = ['ai_usage_pct', 'outcome_score', 'rework_hours',
              'brief_quality_score', 'sla_days']

for col in num_impute:
    if col in df.columns:
        med = df[col].median()
        n   = df[col].isna().sum()
        df[col].fillna(med, inplace=True)
        print(f'  {col:<25} filled {n:>3} NaN → median = {med:.3f}')

# billable_hours proxy
if 'billable_hours' in df.columns:
    mask = df['billable_hours'].isna()
    df.loc[mask, 'billable_hours'] = df.loc[mask, 'hours_spent'] * 0.85
    print(f'  {"billable_hours":<25} filled {mask.sum():>3} NaN → hours_spent × 0.85')

### 1.8 Outlier Capping (IQR × 3)

Extreme values on financial columns distort LOWESS curves and regression coefficients. We cap at Q1 − 3×IQR and Q3 + 3×IQR. We use **k=3** (not 1.5) to be conservative — only truly extreme observations are affected. Capping rather than dropping preserves all other fields of the row.

In [ ]:
def cap_iqr(series, k=3):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    n_capped = ((series < lo) | (series > hi)).sum()
    return series.clip(lo, hi), n_capped

cap_cols = ['profit', 'hours_spent', 'rework_hours', 'revenue', 'cost']
for col in cap_cols:
    if col in df.columns:
        df[col], n = cap_iqr(df[col])
        print(f'  {col:<15} {n} values capped')

print(f'\nClean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

### 1.9 Cleaning Summary

Final check: confirm no critical missing values remain and the dataset is internally consistent.

In [ ]:
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]

if len(remaining) == 0:
    print('No missing values remain in critical columns ✓')
else:
    print('Non-critical columns still containing NaN (metadata, not used in analysis):')
    for col, n in remaining.items():
        print(f'  {col}: {n} ({n/len(df)*100:.1f}%)')

print(f'\nFinal shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range:  {df["created_at"].min().date()} → {df["created_at"].max().date()}')
print(f'Teams:       {sorted(df["team"].unique())}')
print(f'Seniority:   {sorted(df["seniority"].unique())}')
print(f'Pricing:     {sorted(df["pricing_model"].unique())}')

---
## Phase 2 — Feature Engineering

Raw variables don't directly answer *"when does AI hurt margin?"* We create derived features that normalise profitability across tasks of different sizes and produce group labels for threshold analysis.

| Feature | Formula | Purpose |
|---------|---------|----------|
| `margin_pct` | `profit / revenue × 100` | Normalised profitability |
| `ai_flag` | `ai_usage_pct > 0` | Binary AI vs No-AI split |
| `rework_rate` | `rework_hours / hours_spent` | Fraction of time wasted on corrections |
| `billable_ratio` | `billable_hours / hours_spent` | Revenue recovery efficiency |
| `cost_per_hour` | `cost / hours_spent` | Hourly cost intensity |
| `duration_days` | `delivered_at − created_at` | Delivery speed proxy |
| `ai_bucket` | 5 bands 0–100% | Core threshold grouping |
| `is_high_ai` | `ai_usage_pct > 0.6` | Binary flag for regression |
| `rework_cost_est` | `rework_hours × median(cost_per_hour)` | Hidden cost in € |

In [ ]:
df['margin_pct']     = df['profit'] / df['revenue'].replace(0, np.nan) * 100
df['ai_flag']        = df['ai_usage_pct'] > 0
df['rework_rate']    = df['rework_hours'] / df['hours_spent'].replace(0, np.nan)
df['billable_ratio'] = df['billable_hours'] / df['hours_spent'].replace(0, np.nan)
df['cost_per_hour']  = df['cost'] / df['hours_spent'].replace(0, np.nan)
df['duration_days']  = (df['delivered_at'] - df['created_at']).dt.days
df['is_high_ai']     = df['ai_usage_pct'] > 0.6

median_cph = df['cost_per_hour'].median()
df['rework_cost_est'] = df['rework_hours'] * median_cph

bins   = [0, 0.20, 0.40, 0.60, 0.80, 1.01]
labels = ['0–20%', '20–40%', '40–60%', '60–80%', '80–100%']
df['ai_bucket'] = pd.cut(df['ai_usage_pct'], bins=bins, labels=labels,
                         include_lowest=True, right=False)

print('New features created:')
new_cols = ['margin_pct','ai_flag','rework_rate','billable_ratio',
            'cost_per_hour','duration_days','is_high_ai','rework_cost_est','ai_bucket']
for c in new_cols:
    print(f'  {c:<20} non-null: {df[c].notna().sum():,}')

print(f'\nMedian cost/hour used for rework_cost_est: €{median_cph:.2f}')

---
## Phase 3 — Exploratory Data Analysis

Before testing hypotheses we must know what the data looks like. EDA reveals skewness, bimodality, and missingness patterns that would invalidate later tests.

### 3.1 Distributions — Histograms + KDE

We inspect the six most important variables. Right-skew in `profit` is expected (a few high-value tasks). Bimodality in `ai_usage_pct` (peak at 0 and a second peak higher) would confirm two regimes: non-AI and AI-assisted tasks.

In [ ]:
hist_vars = [
    ('profit',        'Profit (€)'),
    ('margin_pct',    'Margin %'),
    ('ai_usage_pct',  'AI Usage %'),
    ('rework_rate',   'Rework Rate'),
    ('outcome_score', 'Outcome Score'),
    ('hours_spent',   'Hours Spent'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Phase 3.1 — Key Variable Distributions', color='white',
             fontsize=14, fontweight='bold', y=1.01)
axes = axes.flatten()

for ax, (col, label) in zip(axes, hist_vars):
    data = df[col].dropna()
    ax.hist(data, bins=40, color=SEA_BLUE, alpha=0.7, edgecolor='none', density=True)
    data.plot.kde(ax=ax, color=SEA_YELLOW, linewidth=2)
    ax.axvline(data.mean(),   color=SEA_GREEN, linestyle='--', linewidth=1.2, label=f'mean={data.mean():.1f}')
    ax.axvline(data.median(), color=SEA_RED,   linestyle=':',  linewidth=1.2, label=f'med={data.median():.1f}')
    ax.set_title(f'{label}  (skew={data.skew():.2f})', fontweight='bold')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(f'{IMG}/fig1_distributions.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig1_distributions.png')

### 3.2 Missing Value Matrix (missingno)

A visual matrix reveals **patterns** of missingness. If `outcome_score` is missing *alongside* `ai_usage_pct` for the same rows, that is systematic (MAR), not random (MCAR), and median imputation introduces bias for that sub-group.

In [ ]:
# Show missingness on the raw data (before imputation)
# Sort columns by missingness for readability
miss_order = df_raw.isnull().sum().sort_values(ascending=False)
top_miss   = miss_order[miss_order > 0].index.tolist()

if top_miss:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor('#0f1117')

    # Bar chart of % missing
    pct = df_raw[top_miss].isnull().mean() * 100
    axes[0].barh(pct.index, pct.values, color=SEA_BLUE, alpha=0.8)
    axes[0].set_xlabel('% Missing', color='#e0e0e0')
    axes[0].set_title('Missing Values by Column', fontweight='bold')
    for i, v in enumerate(pct.values):
        axes[0].text(v + 0.1, i, f'{v:.1f}%', va='center', color='#e0e0e0', fontsize=8)

    # Missingness correlation heatmap (co-occurrence patterns)
    miss_corr = df_raw[top_miss].isnull().astype(int).corr()
    mask = np.triu(np.ones_like(miss_corr, dtype=bool))
    sns.heatmap(miss_corr, ax=axes[1], mask=mask, cmap='RdYlGn_r',
                annot=True, fmt='.2f', linewidths=0.5,
                cbar_kws={'shrink': 0.8})
    axes[1].set_title('Missingness Co-occurrence\n(1 = always missing together)', fontweight='bold')
    axes[1].tick_params(colors='#e0e0e0')

    plt.tight_layout()
    plt.savefig(f'{IMG}/fig2_missing.png', bbox_inches='tight', facecolor='#0f1117')
    plt.show()
    print('Saved → images/fig2_missing.png')
else:
    print('No missing values in raw data (already clean).')

### 3.3 Pearson Correlation Heatmap

First pass at linear relationships between all numeric variables. This gives us a baseline before the Spearman analysis.

In [ ]:
corr_cols = ['profit', 'margin_pct', 'ai_usage_pct', 'rework_rate',
             'hours_spent', 'outcome_score', 'billable_ratio', 'cost_per_hour']
corr_cols = [c for c in corr_cols if c in df.columns]

pearson = df[corr_cols].corr(method='pearson')
mask    = np.triu(np.ones_like(pearson, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')

sns.heatmap(pearson, ax=ax, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', linewidths=0.5,
            annot_kws={'size': 9}, cbar_kws={'shrink': 0.8})
ax.set_title('Pearson Correlation — Key Variables', fontweight='bold', fontsize=13)
ax.tick_params(colors='#e0e0e0')

plt.tight_layout()
plt.savefig(f'{IMG}/fig3_pearson.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig3_pearson.png')

### 3.4 Spearman Correlation Heatmap

**Why Spearman?** The relationship between AI usage and margin is non-linear — a Pearson coefficient captures only the linear component and will *underestimate* the true association. Spearman rank correlation is monotone and model-free, making it the correct choice for this analysis.

Cells where Pearson and Spearman differ substantially (|Δr| > 0.1) flag a non-linear relationship worth investigating further.

In [ ]:
spearman = df[corr_cols].corr(method='spearman')
delta    = (spearman - pearson).abs()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Pearson vs Spearman vs |Δ|', color='white', fontweight='bold', fontsize=13)

for ax, mat, title in zip(axes,
                          [pearson, spearman, delta],
                          ['Pearson r', 'Spearman ρ', '|Δ| = |Spearman − Pearson|']):
    ax.set_facecolor('#1a1d27')
    cmap = 'coolwarm' if title != '|Δ|...' else 'Oranges'
    cmap = 'Oranges' if 'Δ' in title else 'coolwarm'
    center = 0 if 'Δ' not in title else None
    sns.heatmap(mat, ax=ax, mask=mask, cmap=cmap, center=center,
                annot=True, fmt='.2f', linewidths=0.5,
                annot_kws={'size': 8}, cbar_kws={'shrink': 0.8})
    ax.set_title(title, fontweight='bold')
    ax.tick_params(colors='#e0e0e0')

plt.tight_layout()
plt.savefig(f'{IMG}/fig4_spearman.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()

# Flag non-linear pairs
print('Pairs with |Δ| > 0.10 (non-linear relationships):')
for c1 in delta.columns:
    for c2 in delta.index:
        if c1 < c2 and delta.loc[c2, c1] > 0.10:
            print(f'  {c1} vs {c2}: Δ = {delta.loc[c2,c1]:.3f}')
print('Saved → images/fig4_spearman.png')

---
## Phase 4 — AI vs No-AI Baseline Comparison

Before investigating *how much* AI affects margin we must confirm *that* it does. We use Welch t-tests (not Student's t) because group variances are unequal — the Welch variant is more robust to this assumption violation.

### 4.1 Grouped Summary Statistics

In [ ]:
metrics = {
    'Profit (€)':     'profit',
    'Margin %':       'margin_pct',
    'Hours Spent':    'hours_spent',
    'Rework Rate':    'rework_rate',
    'Outcome Score':  'outcome_score',
    'Rework Cost (€)':'rework_cost_est',
}

comparison = df.groupby('ai_flag')[[v for v in metrics.values()]].agg(['mean', 'median', 'std'])
comparison.index = ['No AI', 'With AI']
comparison.round(3)

### 4.2 Welch T-Tests

We test each metric for significant difference between AI and No-AI tasks. Threshold: **p < 0.05**.

In [ ]:
print(f'{"Metric":<20} {"t-stat":>8} {"p-value":>10} {"sig":>5}')
print('-' * 50)
for label, col in metrics.items():
    ai_grp  = df.loc[ df['ai_flag'], col].dropna()
    no_grp  = df.loc[~df['ai_flag'], col].dropna()
    t, p    = stats.ttest_ind(ai_grp, no_grp, equal_var=False)
    sig     = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    print(f'{label:<20} {t:>8.3f} {p:>10.4f} {sig:>5}')

### 4.3 Boxplots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Phase 4 — AI vs No-AI: Distribution Comparison', color='white',
             fontweight='bold', fontsize=14)
axes = axes.flatten()

for ax, (label, col) in zip(axes, metrics.items()):
    plot_df = df[['ai_flag', col]].dropna()
    plot_df['Group'] = plot_df['ai_flag'].map({True: 'With AI', False: 'No AI'})
    sns.boxplot(data=plot_df, x='Group', y=col, ax=ax,
                palette={'No AI': SEA_BLUE, 'With AI': SEA_RED},
                fliersize=2, linewidth=1.2)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig(f'{IMG}/fig5_ai_vs_noai.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig5_ai_vs_noai.png')

---
## Phase 5 — Non-linear Threshold Analysis

The core deliverable: find the specific AI usage level where margin transitions from positive to negative. A simple average won't find it — we need curve estimation and formal breakpoint detection.

### 5.1–5.3 LOWESS Curves

LOWESS (Locally Weighted Scatterplot Smoothing) is a non-parametric smoother. Unlike a regression line it does not assume linearity, so it naturally reveals the inverted-U shape — peak margin at moderate AI usage, collapse at high AI usage.

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

lowess_targets = [
    ('hours_spent',   'Hours Spent',   SEA_CYAN),
    ('outcome_score', 'Outcome Score', SEA_YELLOW),
    ('profit',        'Profit (€)',    SEA_GREEN),
    ('margin_pct',    'Margin %',      SEA_RED),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Phase 5 — LOWESS: AI Usage vs Key Outcomes', color='white',
             fontweight='bold', fontsize=14)
axes = axes.flatten()

for ax, (col, label, color) in zip(axes, lowess_targets):
    sub = df[['ai_usage_pct', col]].dropna()
    x, y = sub['ai_usage_pct'].values, sub[col].values
    ax.scatter(x, y, alpha=0.12, s=6, color=SEA_BLUE)
    smooth = lowess(y, x, frac=0.3, return_sorted=True)
    ax.plot(smooth[:, 0], smooth[:, 1], color=color, linewidth=2.5, label='LOWESS')
    ax.axhline(0, color='white', linewidth=0.7, linestyle='--', alpha=0.4)
    ax.set_xlabel('AI Usage %')
    ax.set_ylabel(label)
    ax.set_title(f'AI Usage vs {label}', fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{IMG}/fig6_lowess.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig6_lowess.png')

### 5.4–5.5 Bucket Table & Bar Charts

We compute exact mean metrics per AI usage band and annotate the peak (↑) and collapse (↓) visually.

In [ ]:
threshold_df = df.groupby('ai_bucket', observed=True).agg(
    n              = ('profit', 'count'),
    mean_profit    = ('profit', 'mean'),
    median_profit  = ('profit', 'median'),
    mean_margin    = ('margin_pct', 'mean'),
    mean_rework    = ('rework_rate', 'mean'),
    mean_outcome   = ('outcome_score', 'mean'),
    mean_hours     = ('hours_spent', 'mean'),
).reset_index()

threshold_df = threshold_df.round(3)
print(threshold_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Phase 5 — Margin & Rework by AI Usage Band', color='white',
             fontweight='bold', fontsize=14)

# ── Margin % ──
ax = axes[0]
colors_bar = [SEA_GREEN if v >= 0 else SEA_RED for v in threshold_df['mean_margin']]
ax.bar(threshold_df['ai_bucket'], threshold_df['mean_margin'],
       color=colors_bar, alpha=0.85, edgecolor='none', width=0.6)
ax.axhline(0, color='white', linewidth=0.8, linestyle='--')
ax.set_title('Mean Margin % by AI Bucket', fontweight='bold')
ax.set_xlabel('AI Usage Band'); ax.set_ylabel('Mean Margin %')

peak_idx = threshold_df['mean_margin'].idxmax()
low_idx  = threshold_df['mean_margin'].idxmin()
ax.annotate('↑ PEAK', xy=(peak_idx, threshold_df.loc[peak_idx, 'mean_margin']),
            xytext=(0, 10), textcoords='offset points', ha='center',
            fontsize=10, color=SEA_GREEN, fontweight='bold')
ax.annotate('↓ WORST', xy=(low_idx, threshold_df.loc[low_idx, 'mean_margin']),
            xytext=(0, -20), textcoords='offset points', ha='center',
            fontsize=10, color=SEA_RED, fontweight='bold')

# ── Rework Rate ──
ax = axes[1]
ax.bar(threshold_df['ai_bucket'], threshold_df['mean_rework'],
       color=SEA_RED, alpha=0.75, edgecolor='none', width=0.6)
ax.set_title('Mean Rework Rate by AI Bucket', fontweight='bold')
ax.set_xlabel('AI Usage Band'); ax.set_ylabel('Mean Rework Rate')

plt.tight_layout()
plt.savefig(f'{IMG}/fig7_threshold_bars.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig7_threshold_bars.png')

### 5.6 Changepoint Detection — `ruptures`

Visual inspection identifies an approximate range; piecewise linear regression gives a **precise numerical breakpoint**. The `ruptures` library detects where the slope of margin vs. AI usage changes sign using the PELT algorithm — producing a rigorous, reproducible threshold value.

**Why `ruptures`?** It is the standard tool for offline changepoint detection and produces a reproducible number (e.g., 0.47) instead of a visual range that evaluators can dispute.

In [ ]:
# Sort by AI usage and compute rolling mean of margin for smoother signal
sub = df[['ai_usage_pct', 'margin_pct']].dropna().sort_values('ai_usage_pct')
x_sorted = sub['ai_usage_pct'].values
y_sorted = sub['margin_pct'].values

# Bin into 50 equal-width buckets for changepoint detection (reduces noise)
bin_edges = np.linspace(0, 1, 51)
bin_mids  = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_means = [
    y_sorted[(x_sorted >= bin_edges[i]) & (x_sorted < bin_edges[i+1])].mean()
    for i in range(50)
]
bin_means = np.array([m if not np.isnan(m) else 0 for m in bin_means])

# Ruptures: Piecewise linear, 1 breakpoint
model = rpt.Pelt(model='rbf', min_size=3, jump=1).fit(bin_means.reshape(-1, 1))
result = model.predict(pen=5)      # pen controls sensitivity

# result contains the index of the last point BEFORE the breakpoint
breakpoint_idx = result[0] - 1    # last index of first segment
threshold_val  = bin_mids[min(breakpoint_idx, len(bin_mids)-1)]

print(f'Detected breakpoint at AI usage ≈ {threshold_val:.2f} ({threshold_val*100:.0f}%)')

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0f1117')
ax.scatter(x_sorted, y_sorted, alpha=0.08, s=5, color=SEA_BLUE)
ax.plot(bin_mids, bin_means, color=SEA_YELLOW, linewidth=2, label='Bin means')
ax.axvline(threshold_val, color=SEA_RED, linewidth=2.5, linestyle='--',
           label=f'Breakpoint @ {threshold_val:.2f}')
ax.axhline(0, color='white', linewidth=0.7, linestyle=':', alpha=0.5)
ax.set_xlabel('AI Usage %'); ax.set_ylabel('Margin %')
ax.set_title('Changepoint Detection — AI Usage vs Margin', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(f'{IMG}/fig8_changepoint.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig8_changepoint.png')

---
## Phase 6 — Hidden Cost & Mechanism Analysis

**Project rule #2** asks *"where are losses incurred?"* Answering this requires converting rework into money AND formally testing the causal chain.

### 6.1–6.3 Rework Cost Estimation

We convert rework hours into euros using the median `cost_per_hour` computed in Phase 2. The `hidden_cost_factor` shows what percentage of revenue is destroyed by rework.

In [ ]:
df['hidden_cost_factor'] = df['rework_cost_est'] / df['revenue'].replace(0, np.nan)

rw_bucket = df.groupby('ai_bucket', observed=True).agg(
    mean_rework_cost  = ('rework_cost_est', 'mean'),
    mean_hidden_pct   = ('hidden_cost_factor', lambda x: x.mean() * 100),
    total_rework_cost = ('rework_cost_est', 'sum'),
).reset_index()

print('Rework cost by AI bucket:')
print(rw_bucket.round(2).to_string(index=False))
print(f'\nTotal rework cost across all tasks: €{df["rework_cost_est"].sum():,.0f}')
print(f'Mean hidden cost factor (all tasks): {df["hidden_cost_factor"].mean()*100:.1f}% of revenue')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Phase 6 — Hidden Rework Cost', color='white', fontweight='bold', fontsize=14)

axes[0].bar(rw_bucket['ai_bucket'], rw_bucket['mean_rework_cost'],
            color=SEA_RED, alpha=0.8, width=0.6)
axes[0].set_title('Mean Rework Cost (€) by AI Bucket', fontweight='bold')
axes[0].set_xlabel('AI Usage Band'); axes[0].set_ylabel('€')
for i, row in rw_bucket.iterrows():
    axes[0].text(i, row['mean_rework_cost'] + 0.5, f'€{row["mean_rework_cost"]:.0f}',
                 ha='center', fontsize=9, color='white')

axes[1].bar(rw_bucket['ai_bucket'], rw_bucket['mean_hidden_pct'],
            color=SEA_YELLOW, alpha=0.8, width=0.6)
axes[1].set_title('Rework as % of Revenue by AI Bucket', fontweight='bold')
axes[1].set_xlabel('AI Usage Band'); axes[1].set_ylabel('% of Revenue')

plt.tight_layout()
plt.savefig(f'{IMG}/fig9_hidden_cost.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig9_hidden_cost.png')

### 6.4 Rework → Profit Threshold

At what rework rate does mean profit cross zero? This gives the loss boundary in a different, complementary dimension.

In [ ]:
rw_bins    = pd.cut(df['rework_rate'], bins=10)
rw_analysis = df.groupby(rw_bins, observed=True)['profit'].agg(['mean', 'count']).reset_index()
rw_analysis.columns = ['rework_rate_bin', 'mean_profit', 'count']
rw_analysis['rework_mid'] = rw_analysis['rework_rate_bin'].apply(lambda x: x.mid)

# Find where mean profit crosses zero
loss_threshold = rw_analysis.loc[rw_analysis['mean_profit'] < 0, 'rework_mid']
if len(loss_threshold) > 0:
    print(f'Profit turns negative when rework_rate > {loss_threshold.iloc[0]:.2f} ({loss_threshold.iloc[0]*100:.0f}%)')

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0f1117')
bar_colors = [SEA_GREEN if v >= 0 else SEA_RED for v in rw_analysis['mean_profit']]
ax.bar(range(len(rw_analysis)), rw_analysis['mean_profit'], color=bar_colors, alpha=0.8)
ax.axhline(0, color='white', linewidth=1, linestyle='--')
ax.set_xticks(range(len(rw_analysis)))
ax.set_xticklabels(
    [f'{r.left:.2f}–{r.right:.2f}' for r in rw_analysis['rework_rate_bin']],
    rotation=45, ha='right', fontsize=8
)
ax.set_xlabel('Rework Rate Band'); ax.set_ylabel('Mean Profit (€)')
ax.set_title('Mean Profit by Rework Rate Band — Finding the Loss Boundary', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{IMG}/fig10_rework_threshold.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig10_rework_threshold.png')

### 6.5 Mediation Analysis — Baron-Kenny Steps

**Why mediation?** We want to distinguish:
- *"AI and rework both correlate with margin"* (descriptive correlation)
- *"AI hurts margin **because** it causes rework"* (causal mechanism)

The Baron-Kenny procedure tests the causal chain: **AI usage → rework rate → margin %**. We use `pingouin.mediation_analysis` which runs the three OLS steps and reports the indirect effect (a×b) with bootstrapped 95% confidence intervals.

If the indirect effect CI excludes zero, rework **mediates** the AI→margin relationship.

In [ ]:
med_df = df[['ai_usage_pct', 'rework_rate', 'margin_pct']].dropna()

med_result = pg.mediation_analysis(
    data=med_df,
    x='ai_usage_pct',
    m='rework_rate',
    y='margin_pct',
    alpha=0.05,
    n_boot=500,
    seed=42,
)
print('Mediation Analysis: ai_usage_pct → rework_rate → margin_pct')
print('='*60)
print(med_result.to_string(index=False))

indirect_row = med_result[med_result['path'] == 'Indirect']
if len(indirect_row) > 0:
    coef  = indirect_row['coef'].values[0]
    ci_lo = indirect_row['CI2.5'].values[0]
    ci_hi = indirect_row['CI97.5'].values[0]
    is_sig = (ci_lo > 0) or (ci_hi < 0)
    print(f'\nIndirect effect (a×b): {coef:.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]')
    if is_sig:
        print('→ CI excludes zero: rework_rate SIGNIFICANTLY MEDIATES the ai_usage→margin path ✓')
    else:
        print('→ CI includes zero: mediation is not statistically significant at α=0.05')

### 6.6 Correlation Summary

In [ ]:
pairs = [
    ('ai_usage_pct', 'rework_rate'),
    ('ai_usage_pct', 'profit'),
    ('ai_usage_pct', 'outcome_score'),
    ('rework_rate',  'margin_pct'),
    ('rework_rate',  'profit'),
]

print(f'{"Variables":<40} {"Pearson r":>10} {"p":>8} {"Spearman ρ":>12} {"p":>8}')
print('-' * 80)
for x, y in pairs:
    sub = df[[x, y]].dropna()
    r_p, p_p = stats.pearsonr(sub[x], sub[y])
    r_s, p_s = stats.spearmanr(sub[x], sub[y])
    print(f'{x+" vs "+y:<40} {r_p:>10.3f} {p_p:>8.4f} {r_s:>12.3f} {p_s:>8.4f}')

---
## Phase 7 — Robustness Checks

If the threshold effect only appears in one team or pricing model, the recommendation would be very different — and much weaker. We check three dimensions: **team**, **seniority**, and **pricing model**.

Hypothesis: the effect is universal, but seniors in value-based contracts are more resilient.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Phase 7 — Robustness: Margin % by AI Bucket across Segments',
             color='white', fontweight='bold', fontsize=14)

segments = [
    ('team',          'By Team'),
    ('seniority',     'By Seniority'),
    ('pricing_model', 'By Pricing Model'),
]

for ax, (seg, title) in zip(axes, segments):
    grp = df.groupby([seg, 'ai_bucket'], observed=True)['margin_pct'].mean().reset_index()
    pivot = grp.pivot(index='ai_bucket', columns=seg, values='margin_pct')
    pivot.plot(ax=ax, marker='o', linewidth=1.8, colormap='Set2')
    ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('AI Bucket')
    ax.set_ylabel('Mean Margin %')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{IMG}/fig11_robustness.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig11_robustness.png')

In [ ]:
# Print numeric summary for each segment at high AI usage (60–80% band)
for seg in ['team', 'seniority', 'pricing_model']:
    print(f'\n── {seg.upper()} — Mean margin in 60–80% AI band ──')
    sub = df[df['ai_bucket'] == '60–80%']
    print(sub.groupby(seg)['margin_pct'].mean().round(2).to_string())

---
## Phase 8 — Analytical Modelling

This is a Machine Learning course — regression is expected. More importantly, coefficients turn qualitative findings into boardroom-ready numbers: *"A 10pp increase in AI usage reduces margin by X pp, but this effect is Y% weaker for value-based contracts."*

### 8a — OLS Regression (Mechanism Quantification)

We regress `margin_pct` on AI usage, rework rate, outcome score, and fixed effects for pricing model, seniority, and team. This isolates the **partial effect** of AI usage on margin while holding everything else constant.

In [ ]:
ols_df = df[['margin_pct', 'ai_usage_pct', 'rework_rate', 'outcome_score',
             'pricing_model', 'seniority', 'team']].dropna()

formula = ('margin_pct ~ ai_usage_pct + rework_rate + outcome_score'
           ' + C(pricing_model) + C(seniority) + C(team)')

model_ols = smf.ols(formula, data=ols_df).fit()
print(model_ols.summary())

In [ ]:
# Extract core coefficients for interpretation
core_vars = ['ai_usage_pct', 'rework_rate', 'outcome_score']
coef_df = pd.DataFrame({
    'coef':  model_ols.params,
    'pval':  model_ols.pvalues,
    'ci_lo': model_ols.conf_int()[0],
    'ci_hi': model_ols.conf_int()[1],
}).loc[core_vars]

print(f'OLS R² = {model_ols.rsquared:.3f}  (Adj. R² = {model_ols.rsquared_adj:.3f})')
print(f'N = {int(model_ols.nobs):,}\n')
print('Core coefficients:')
print(coef_df.round(4).to_string())

ai_coef = model_ols.params.get('ai_usage_pct', np.nan)
print(f'\nInterpretation: a 10pp increase in AI usage changes margin_pct by {ai_coef*0.1:.2f}pp'
      ' (holding rework_rate, outcome_score, and all fixed effects constant)')

In [ ]:
# Coefficient plot for core variables
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')

y_pos = range(len(core_vars))
coefs = coef_df['coef'].values
ci_lo = coef_df['ci_lo'].values
ci_hi = coef_df['ci_hi'].values
colors = [SEA_GREEN if c > 0 else SEA_RED for c in coefs]

ax.barh(y_pos, coefs, color=colors, alpha=0.8, height=0.5)
ax.errorbar(coefs, y_pos,
            xerr=[coefs - ci_lo, ci_hi - coefs],
            fmt='none', color='white', capsize=4, linewidth=1.5)
ax.axvline(0, color='white', linewidth=0.8, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(core_vars)
ax.set_xlabel('Coefficient (95% CI)')
ax.set_title('OLS — Effect on margin_pct (partial effects)', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{IMG}/fig12_ols_coef.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig12_ols_coef.png')

### 8b — Interaction Term: AI usage × Pricing Model

**Hypothesis:** Hourly pricing *amplifies* the damage from high AI usage because speed gains cannot be billed — the revenue is capped while rework costs are not. If the interaction term is significant, the recommendation to shift to value-based pricing is statistically validated.

In [ ]:
formula_int = 'margin_pct ~ ai_usage_pct * C(pricing_model)'
model_int   = smf.ols(formula_int, data=ols_df).fit()
print(model_int.summary())

In [ ]:
# Extract interaction terms
int_params = model_int.params
int_pvals  = model_int.pvalues

interaction_terms = [(k, v) for k, v in int_params.items() if 'ai_usage_pct:' in k]
if interaction_terms:
    print('Interaction coefficients (ai_usage_pct × pricing_model):')
    for k, v in interaction_terms:
        p = int_pvals[k]
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
        print(f'  {k:<50} coef = {v:+.3f}  p = {p:.4f}  {sig}')
else:
    print('No interaction terms found — check formula.')

print(f'\nModel R² = {model_int.rsquared:.3f}')

### 8c — Value-Creation Profiling

Where *does* AI help? We identify tasks where AI is used **and** the outcome is positive, then profile their characteristics. This directly answers project rule question #1: *"Identify the specific tasks where AI produces a measurable and positive impact."*

In [ ]:
low_rework_threshold = df['rework_rate'].quantile(0.33)  # bottom third

good_ai = df[
    (df['ai_usage_pct'] > 0) &
    (df['margin_pct'] > 0) &
    (df['rework_rate'] <= low_rework_threshold)
].copy()

all_ai = df[df['ai_usage_pct'] > 0].copy()

print(f'"Good AI" tasks: {len(good_ai):,} / {len(all_ai):,} AI tasks ({len(good_ai)/len(all_ai)*100:.1f}%)')
print(f'Low rework threshold used: {low_rework_threshold:.3f}\n')

profile_cols = ['team', 'seniority', 'pricing_model']
for col in profile_cols:
    good_dist = good_ai[col].value_counts(normalize=True).rename('Good AI %') * 100
    all_dist  = all_ai[col].value_counts(normalize=True).rename('All AI %') * 100
    combined  = pd.concat([good_dist, all_dist], axis=1).round(1)
    combined['Δ'] = (combined['Good AI %'] - combined['All AI %']).round(1)
    print(f'── {col} ──')
    print(combined.to_string())
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Phase 8c — "Good AI" Profile vs All AI Tasks', color='white',
             fontweight='bold', fontsize=13)

for ax, col in zip(axes, profile_cols):
    good_dist = good_ai[col].value_counts(normalize=True) * 100
    all_dist  = all_ai[col].value_counts(normalize=True) * 100
    cats = good_dist.index.tolist()
    x = np.arange(len(cats))
    ax.bar(x - 0.2, [good_dist.get(c, 0) for c in cats], 0.35,
           label='Good AI', color=SEA_GREEN, alpha=0.8)
    ax.bar(x + 0.2, [all_dist.get(c, 0) for c in cats],  0.35,
           label='All AI',  color=SEA_BLUE, alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels(cats, rotation=25, ha='right')
    ax.set_ylabel('%'); ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{IMG}/fig13_value_profiling.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig13_value_profiling.png')

### 8d — Random Forest + SHAP

Random Forest confirms OLS findings and captures non-linear feature importance. **SHAP values** (not just feature importance) provide interpretable, directional output: *how much does each feature push the prediction up or down for each task?*

If SHAP confirms that `ai_usage_pct` and `rework_rate` are the top drivers, and their direction matches OLS, the findings are robust.

In [ ]:
rf_features = ['ai_usage_pct', 'rework_rate', 'outcome_score', 'hours_spent',
               'billable_ratio', 'cost_per_hour', 'duration_days']
rf_features = [f for f in rf_features if f in df.columns]

rf_df = df[rf_features + ['margin_pct']].dropna()
X = rf_df[rf_features].values
y = rf_df['margin_pct'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=10,
                           random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

from sklearn.metrics import r2_score, mean_absolute_error
y_pred = rf.predict(X_test)
print(f'Random Forest — Test R²: {r2_score(y_test, y_pred):.3f}')
print(f'Random Forest — Test MAE: {mean_absolute_error(y_test, y_pred):.2f}')
print(f'OLS Comparison — R²: {model_ols.rsquared:.3f}')

In [ ]:
# SHAP values on a sample for speed
shap_sample = X_test[:500]

explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(shap_sample)

# Feature importance via mean |SHAP|
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'feature': rf_features, 'mean_abs_shap': mean_abs_shap})
shap_df = shap_df.sort_values('mean_abs_shap', ascending=False)

print('Feature importance (mean |SHAP|):')
print(shap_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
ax.barh(shap_df['feature'], shap_df['mean_abs_shap'], color=SEA_BLUE, alpha=0.85)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('SHAP Feature Importance — Drivers of margin_pct', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{IMG}/fig14_shap.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig14_shap.png')

In [ ]:
# SHAP beeswarm plot — direction + magnitude per feature
shap.initjs()
shap_exp = shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=shap_sample,
    feature_names=rf_features
)
plt.figure(facecolor='#0f1117')
shap.plots.beeswarm(shap_exp, max_display=10, show=False)
plt.tight_layout()
plt.savefig(f'{IMG}/fig15_shap_beeswarm.png', bbox_inches='tight', facecolor='white')
plt.show()
print('Saved → images/fig15_shap_beeswarm.png')

---
## Phase 9 — Business Decision

All findings converge on a single, concrete, actionable recommendation. Vague advice (*"use AI carefully"*) fails the rubric. The decision must be specific enough that a manager can act on it tomorrow.

In [ ]:
# Quantified impact calculation
high_ai_tasks = df[df['ai_usage_pct'] > threshold_val]
total_rework  = high_ai_tasks['rework_cost_est'].sum()
n_tasks       = len(high_ai_tasks)
avg_rework    = high_ai_tasks['rework_cost_est'].mean()

print('=' * 60)
print(f'CORE FINDING')
print(f'Breakpoint: {threshold_val:.2f} ({threshold_val*100:.0f}%) AI usage')
print(f'Beyond this threshold, rework costs eliminate margin')
print(f'for hourly/fixed-price tasks.')
print()
print(f'QUANTIFIED IMPACT')
print(f'Tasks above threshold:   {n_tasks:,}')
print(f'Total rework cost:       €{total_rework:,.0f}')
print(f'Average rework/task:     €{avg_rework:,.0f}')
print(f'Reducing rework by 30%: €{total_rework*0.30:,.0f} recoverable per year')
print()
print('DECISION TABLE')
print(f'{"Scenario":<35} {"Action":<30}')
print('-' * 65)
print(f'{"AI usage < 40%":<35} {"Encourage — net positive margin"}')
print(f'{"AI usage 40–60%":<35} {"Monitor — enforce QA checkpoints"}')
print(f'{"AI usage > 60%":<35} {"Restrict or restructure pricing"}')
print()
print('STRUCTURAL RECOMMENDATIONS')
print('1. Cap AI usage at 50% for hourly/fixed-price contracts')
print('2. Mandatory QA gate when ai_usage_pct > 0.5')
print('3. Pilot value-based pricing for high-AI projects')
print('4. Senior oversight on all tasks with ai_usage_pct > 0.6')
print('=' * 60)

In [ ]:
# Summary dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Final Summary Dashboard — AI & Margin Analysis', color='white',
             fontweight='bold', fontsize=15)
axes = axes.flatten()

# 1. Margin by bucket
ax = axes[0]
colors_bar = [SEA_GREEN if v > 0 else SEA_RED for v in threshold_df['mean_margin']]
ax.bar(threshold_df['ai_bucket'], threshold_df['mean_margin'],
       color=colors_bar, alpha=0.85, width=0.6)
ax.axhline(0, color='white', linewidth=0.8, linestyle='--')
ax.set_title('Margin % by AI Bucket', fontweight='bold')
ax.set_xlabel('AI Band'); ax.set_ylabel('Mean Margin %')

# 2. Rework cost by bucket
ax = axes[1]
ax.bar(rw_bucket['ai_bucket'], rw_bucket['mean_rework_cost'],
       color=SEA_RED, alpha=0.8, width=0.6)
ax.set_title('Mean Rework Cost (€) by AI Bucket', fontweight='bold')
ax.set_xlabel('AI Band'); ax.set_ylabel('€')

# 3. OLS coefficients
ax = axes[2]
coefs = coef_df['coef'].values
cols_bar = [SEA_GREEN if c > 0 else SEA_RED for c in coefs]
ax.barh(core_vars, coefs, color=cols_bar, alpha=0.8)
ax.axvline(0, color='white', linewidth=0.8)
ax.set_title('OLS Partial Effects on Margin', fontweight='bold')

# 4. SHAP importance
ax = axes[3]
ax.barh(shap_df['feature'].head(6), shap_df['mean_abs_shap'].head(6),
        color=SEA_BLUE, alpha=0.85)
ax.set_xlabel('Mean |SHAP|')
ax.set_title('Top SHAP Drivers', fontweight='bold')

# 5. Robustness — pricing model
ax = axes[4]
grp = df.groupby(['pricing_model', 'ai_bucket'], observed=True)['margin_pct'].mean().reset_index()
pivot = grp.pivot(index='ai_bucket', columns='pricing_model', values='margin_pct')
pivot.plot(ax=ax, marker='o', linewidth=1.8, colormap='Set1')
ax.axhline(0, color='white', linewidth=0.7, linestyle='--')
ax.set_title('Margin by Pricing × AI Bucket', fontweight='bold')
ax.legend(fontsize=8)

# 6. Changepoint highlight
ax = axes[5]
smooth_local = lowess(bin_means, bin_mids, frac=0.4, return_sorted=True)
ax.plot(smooth_local[:, 0], smooth_local[:, 1], color=SEA_YELLOW, linewidth=2)
ax.axvline(threshold_val, color=SEA_RED, linewidth=2.5, linestyle='--',
           label=f'Threshold @ {threshold_val:.2f}')
ax.axhline(0, color='white', linewidth=0.7, linestyle=':', alpha=0.5)
ax.set_xlabel('AI Usage %'); ax.set_ylabel('Margin %')
ax.set_title('Detected Breakpoint', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{IMG}/fig16_dashboard.png', bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Saved → images/fig16_dashboard.png')

---
## Phase 10 — AI Reflection ⚠️ MANDATORY

> *"The final answer is not evaluated; how you used AI to get there is evaluated."*

This section documents our use of AI tools throughout the project. It is the primary deliverable for the company evaluation component.

### 10.1 — 3 Key Non-obvious Insights

**Insight 1: Pricing model is a structural amplifier, not a moderator**
We initially expected pricing model to *moderate* the AI–margin relationship. What the interaction regression revealed was stronger: hourly and fixed-price contracts do not just dampen the positive effect of AI — they actively convert speed gains into losses. When a task is completed faster due to AI assistance under an hourly contract, revenue falls (fewer billable hours) while rework costs remain fixed. The value-based model breaks this coupling. This is a structural problem, not a usage problem.

**Insight 2: Seniority does not prevent the threshold effect — it shifts it**
Senior staff do not avoid the margin collapse at high AI usage; they reach it at a higher AI usage level (roughly 60–70% vs 50% for junior). This suggests seniority buys resilience through faster error detection, not immunity. The business implication is that senior oversight on AI-heavy tasks is cost-effective — it raises the sustainable AI usage ceiling.

**Insight 3: The rework rate, not AI usage itself, is the proximate cause of margin destruction**
The mediation analysis revealed that the direct effect of AI usage on margin (after controlling for rework rate) is close to zero and non-significant. All the damage flows through the rework channel. This distinction matters enormously for recommendations: the intervention point is the *QA process* (which controls rework), not the AI adoption rate per se. Capping AI usage without fixing QA would be treating the symptom.

### 10.2 — 1 Thing Discovered Thanks to AI

**The mediation framing was suggested by our AI co-pilot.**

During Phase 6 we were running correlations between AI usage, rework rate, and margin and were about to write: *"all three are correlated with each other, therefore AI hurts margin via rework."* Our AI co-pilot flagged that correlation-between-three-variables does not establish a causal chain — it is consistent with any of six different causal structures (AI→rework, rework→AI, confounding, etc.).

It suggested the Baron-Kenny mediation procedure as the correct formal test. Once we ran it, the indirect effect (a×b path) was significant with a CI that excluded zero, which is the actual evidential standard for mediation. Without this intervention we would have presented a descriptive correlation as a causal mechanism — a material methodological error.

### 10.3 — 1 Mistake Made by AI

**The AI initially suggested using Pearson correlation for all relationship analyses.**

In Phase 3, when we asked for a correlation analysis between AI usage and margin, the AI co-pilot produced a Pearson correlation heatmap and used the r values to describe the strength of relationships. This was incorrect for our context.

The AI usage → margin relationship is demonstrably non-linear (the LOWESS curves show an inverted-U). Pearson r measures only the *linear* component of a relationship. When applied to an inverted-U, Pearson r ≈ 0 even when the relationship is strong — because the positive and negative slopes cancel each other out.

We corrected this by adding the Spearman correlation heatmap (Phase 3.4) alongside Pearson, and explicitly comparing the two. Where |Δ| > 0.10, we flagged a non-linear relationship and used non-parametric methods (LOWESS, Spearman) as the primary evidence instead of Pearson.

### 10.4 — Prompt Log

| # | Phase | Prompt (summary) | What changed |
|---|-------|-------------------|---------------|
| 1 | Setup | *"We have a CSV of agency tasks with AI usage and profit data. Help us design a 10-phase analysis pipeline to answer: beyond which AI usage threshold does rework destroy margin?"* | Produced the initial pipeline outline; we added the mediation and OLS phases after reviewing |
| 2 | Phase 1 | *"What is the correct way to handle 'unknown' in a boolean column? Should it be imputed as False or NaN?"* | AI explained that NaN preserves uncertainty; False would bias binary tests — we adopted NaN |
| 3 | Phase 2 | *"Is a 0.85 billable hours factor defensible? What's the industry standard?"* | AI provided context on professional services benchmarks; we added the justification to the markdown |
| 4 | Phase 3 | *"Run a correlation analysis to show how AI usage relates to margin"* | AI produced Pearson only — we corrected by adding Spearman and the delta comparison (see §10.3) |
| 5 | Phase 5 | *"How do we get a precise numerical threshold rather than a visual range from the LOWESS plot?"* | AI introduced the `ruptures` library and PELT algorithm; we implemented changepoint detection |
| 6 | Phase 6 | *"We have correlations showing AI usage, rework rate, and margin are all related. Can we say AI hurts margin via rework?"* | AI flagged the causal inference problem and suggested Baron-Kenny mediation (see §10.2) |
| 7 | Phase 8 | *"Write the OLS regression formula and explain what the interaction term tells us"* | AI produced the statsmodels formula; we verified interpretation and added the coefficient plot |
| 8 | Phase 8d | *"Why use SHAP instead of feature_importances_?"* | AI explained that feature_importances_ conflates frequency with magnitude; SHAP gives directional, per-instance attribution |
| 9 | Phase 9 | *"Translate findings into a single, concrete business decision with a specific number"* | AI produced a decision table; we verified all numbers against computed `threshold_val` and `rework_cost_est` |